# Step 1: Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_csv('/kaggle/input/datasets/datascikhan/e-commerce-delivery-and-shipping-data-2026/E-commerce_Delivery_Shipping_Data_2026.csv')

df.head(20)

,order_id,order_date,customer_id,customer_segment,customer_city,customer_country,warehouse_id,warehouse_city,product_category,product_weight_kg,...,package_size,payment_method,order_priority,weather_condition,customer_rating,return_requested,return_reason,delivery_attempts,warehouse_processing_hours,tracking_status
0,ORD-000001,2026-11-15,CUS-002353,Small Business,Tokyo,Japan,WH-004,Toronto,Home & Kitchen,8.23,...,Large,Digital Wallet,Normal,Snow,3,Yes,Late Delivery,2,25.1,Delivered
1,ORD-000002,2026-04-08,CUS-003258,Consumer,Vancouver,Canada,WH-007,Paris,Beauty,0.10,...,Small,Debit Card,Normal,Cloudy,5,No,NaN,1,25.9,Delivered
2,ORD-000003,2026-01-05,CUS-001221,Consumer,Houston,United States,WH-005,London,Home & Kitchen,5.51,...,Large,Debit Card,Low,Snow,4,Yes,Product Defect,1,31.5,Delivered
3,ORD-000004,2026-05-29,CUS-001717,Small Business,Delhi,India,WH-002,Los Angeles,Grocery,3.75,...,Large,Debit Card,Normal,Snow,3,No,NaN,1,12.6,In Transit
4,ORD-000005,2026-02-22,CUS-000716,Small Business,Nagoya,Japan,WH-006,Berlin,Beauty,0.10,...,Small,Credit Card,Normal,Storm,2,No,NaN,1,24.9,Delivered
5,ORD-000006,2026-03-06,CUS-005749,Consumer,Brisbane,Australia,WH-002,Los Angeles,Toys,0.76,...,Medium,Debit Card,Low,Extreme Heat,3,No,NaN,1,32.2,Delivered
6,ORD-000007,2026-07-10,CUS-002409,Small Business,Ottawa,Canada,WH-003,Chicago,Grocery,4.81,...,Large,Bank Transfer,Normal,Rain,3,No,NaN,1,21.0,Delivered
7,ORD-000008,2026-12-03,CUS-003537,Consumer,Los Angeles,United States,WH-004,Toronto,Electronics,3.51,...,Large,Cash on Delivery,Normal,Snow,2,No,NaN,1,21.2,Delivered
8,ORD-000009,2026-12-18,CUS-004446,Consumer,Phoenix,United States,WH-002,Los Angeles,Grocery,2.09,...,Large,Credit Card,High,Clear,4,No,NaN,1,12.7,Delivered
9,ORD-000010,2026-11-05,CUS-005134,Enterprise,Dusseldorf,Germany,WH-002,Los Angeles,Home & Kitchen,0.10,...,Small,Debit Card,Normal,Rain,2,No,NaN,1,15.1,Delivered


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   order_id                    50000 non-null  object 
 1   order_date                  50000 non-null  object 
 2   customer_id                 50000 non-null  object 
 3   customer_segment            50000 non-null  object 
 4   customer_city               50000 non-null  object 
 5   customer_country            50000 non-null  object 
 6   warehouse_id                50000 non-null  object 
 7   warehouse_city              50000 non-null  object 
 8   product_category            50000 non-null  object 
 9   product_weight_kg           50000 non-null  float64
 10  order_value_usd             50000 non-null  float64
 11  shipping_method             50000 non-null  object 
 12  carrier                     50000 non-null  object 
 13  distance_km                 500

# Step 2: Data Cleaning and Preparation

In [4]:
print(df["return_requested"].value_counts())

return_requested
No     41375
Yes     8625
Name: count, dtype: int64


In [5]:
df["return_reason"] = df["return_reason"].fillna("Non Concerné")

In [6]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

print(df["order_date"].dtypes)

datetime64[ns]


In [7]:
df["late_delivery"] = df["late_delivery"].map({"Yes": True, "No": False})
df["return_requested"] = df["return_requested"].map({"Yes": True, "No": False})

In [8]:
colonnes_positives = [
   "product_weight_kg",
    "order_value_usd",
    "shipping_cost_usd",
    "distance_km",
    "warehouse_processing_hours" 
]

print((df[colonnes_positives] <= 0).sum())

product_weight_kg             0
order_value_usd               0
shipping_cost_usd             0
distance_km                   0
warehouse_processing_hours    0
dtype: int64


In [9]:
print(df["customer_rating"].value_counts().sort_index())

customer_rating
1      962
2     9621
3    15872
4    16953
5     6592
Name: count, dtype: int64


In [10]:
print(df["delivery_delay_days"].describe())

count    50000.000000
mean         1.281080
std          1.679527
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         13.000000
Name: delivery_delay_days, dtype: float64


# Step 3 : EDA (Exploratory Data Analysis)

In [11]:
fig = px.histogram(
    df, 
    x="delivery_delay_days", 
    nbins=15, 
    title="Distribution des jours de retard de livraison",
    labels={"delivery_delay_days": "Jours de retard", "count": "Nombre de commandes"},
    color_discrete_sequence=["#2b5c8f"]
)
fig.update_layout(bargap=0.1)
fig.show()

In [12]:
fig = px.scatter(
    df.sample(5000),
    x="distance_km", 
    y="shipping_cost_usd", 
    color="package_size",
    size="product_weight_kg",
    title="Coût d'expédition en fonction de la distance et de la taille du colis",
    labels={"distance_km": "Distance (km)", "shipping_cost_usd": "Coût d'expédition ($)"}
)
fig.show()

In [13]:
rating_by_delay = df.groupby("late_delivery")["customer_rating"].mean().reset_index()
rating_by_delay["late_delivery"] = rating_by_delay["late_delivery"].map({True: "En retard", False: "À l'heure"})

fig = px.bar(
    rating_by_delay, 
    x="late_delivery", 
    y="customer_rating", 
    title="Impact du retard de livraison sur la note moyenne client",
    labels={"late_delivery": "Statut de livraison", "customer_rating": "Note moyenne (sur 5)"},
    color="late_delivery",
    color_discrete_map={"À l'heure": "#2ecc71", "En retard": "#e74c3c"}
)
fig.update_yaxes(range=[0, 5])
fig.show()

In [14]:
df_returns = df[df["return_requested"] == True]

return_reasons_counts = df_returns["return_reason"].value_counts().reset_index()
return_reasons_counts.columns = ["return_reason", "count"]

fig = px.pie(
    return_reasons_counts, 
    names="return_reason", 
    values="count", 
    title="Répartition des motifs de retour produit",
    hole=0.4
)
fig.show()

# Step 4: Calculation of Key Performance Indicators (KPIs)

In [15]:
# ==========================================
# 1. TOP GLOBAL KPIS
# ==========================================

total_orders = len(df)
on_time_count = (df["late_delivery"] == False).sum()
on_time_percentage = (on_time_count / total_orders) * 100
average_delay = df["delivery_delay_days"].mean()

# Ratio Coût de transport / Valeur de la commande (en %)
df["cost_value_ratio"] = (df["shipping_cost_usd"] / df["order_value_usd"]) * 100
average_cost_ratio = df["cost_value_ratio"].mean()

print("==========================================")
print("             TOP GLOBAL KPIS              ")
print("==========================================")
print(f"📦 Total Orders       : {total_orders:,}")
print(f"⏱️ On-Time Delivery   : {on_time_percentage:.2f}%")
print(f"⏳ Average Delay Days : {average_delay:.2f} days")
print(f"💸 Cost / Value Ratio : {average_cost_ratio:.2f}%\n")


# ==========================================
# 2. SECTION A : LOGISTICS PERFORMANCE
# ==========================================

carrier_delay = df.groupby("carrier")["late_delivery"].mean().reset_index()
carrier_delay["late_delivery"] = carrier_delay["late_delivery"] * 100
carrier_delay.columns = ["Carrier", "Late_Rate_Pct"]
print("--- Section A : Late Rate by Carrier ---")
print(carrier_delay.sort_values(by="Late_Rate_Pct", ascending=False))
print("\n")


# ==========================================
# 3. SECTION B : COSTS & CARRIERS
# ==========================================

shipping_method_counts = (df["shipping_method"].value_counts(normalize=True) * 100).reset_index()
shipping_method_counts.columns = ["Shipping_Method", "Percentage"]
print("--- Section B : Shipping Methods Breakdown ---")
print(shipping_method_counts)
print("\n")


# ==========================================
# 4. SECTION C : RETURNS & CUSTOMER QUALITY
# ==========================================

df_returns = df[df["return_requested"] == True]
return_reasons = (df_returns["return_reason"].value_counts(normalize=True) * 100).reset_index()
return_reasons.columns = ["Return_Reason", "Percentage"]
print("--- Section C : Main Return Reasons ---")
print(return_reasons)
print("\n")


# ==========================================
# 5. SECTION D : EXTERNAL FACTORS
# ==========================================

weather_impact = df.groupby("weather_condition").agg(
    avg_attempts=("delivery_attempts", "mean"),
    on_time_pct=("late_delivery", lambda x: (x == False).mean() * 100)
).reset_index()

print("--- Section D : Weather Impact ---")
print(weather_impact)

             TOP GLOBAL KPIS              
📦 Total Orders       : 50,000
⏱️ On-Time Delivery   : 43.68%
⏳ Average Delay Days : 1.28 days
💸 Cost / Value Ratio : 309.51%

--- Section A : Late Rate by Carrier ---
               Carrier  Late_Rate_Pct
3        GlobalExpress      65.751886
5        PrimeDelivery      61.455133
0            BlueRoute      61.025641
2  FastTrack Logistics      56.571146
1         EagleCourier      54.868823
4            ParcelPro      52.055224
6          SpeedyCargo      51.526291
7            SwiftShip      47.344778


--- Section B : Shipping Methods Breakdown ---
  Shipping_Method  Percentage
0        Standard      34.054
1   International      31.678
2         Express      19.392
3         Economy      11.154
4        Same Day       3.722


--- Section C : Main Return Reasons ---
      Return_Reason  Percentage
0    Product Defect   25.321739
1   Damaged Package   17.518841
2  Not as Described   15.223188
3      Changed Mind   14.921739
4        Wrong It

# Step 5: Development of Interactive Visualizations (Plotly)

In [16]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import plotly.express as px

# ==========================================
# DATA PREPARATION & INITIALIZATION
# ==========================================

# Ensure order_date is datetime and calculate auxiliary ratio
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["cost_value_ratio"] = (df["shipping_cost_usd"] / df["order_value_usd"]) * 100

# Define interactive widgets for global filters
category_dropdown = widgets.SelectMultiple(
    options=sorted(df["product_category"].unique().tolist()),
    value=list(df["product_category"].unique()),
    description="Category:",
)

shipping_dropdown = widgets.SelectMultiple(
    options=sorted(df["shipping_method"].unique().tolist()),
    value=list(df["shipping_method"].unique()),
    description="Shipping:",
)

warehouse_dropdown = widgets.SelectMultiple(
    options=sorted(df["warehouse_id"].unique().tolist()),
    value=list(df["warehouse_id"].unique()),
    description="Warehouse:",
)

# Output container for the dashboard display
output = widgets.Output()


def update_dashboard(change):
  with output:
    output.clear_output(wait=True)

    # Filter dataframe based on widget selections
    filtered_df = df[
        df["product_category"].isin(category_dropdown.value)
        & df["shipping_method"].isin(shipping_dropdown.value)
        & df["warehouse_id"].isin(warehouse_dropdown.value)
    ]

    if filtered_df.empty:
      print("No data available for the selected filters.")
      return

    # ==========================================
    # TOP GLOBAL KPIS CALCULATION
    # ==========================================
    total_orders = len(filtered_df)
    on_time_pct = (
        (filtered_df["late_delivery"] == False).sum() / total_orders
    ) * 100
    avg_delay = filtered_df["delivery_delay_days"].mean()
    avg_cost_ratio = filtered_df["cost_value_ratio"].mean()

    print(
        f"📦 TOTAL ORDERS: {total_orders:,} | ⏱️ ON-TIME: {on_time_pct:.1f}% |"
        f" ⏳ AVG DELAY: {avg_delay:.1f} days | 💸 COST/VALUE RATIO:"
        f" {avg_cost_ratio:.1f}%\n"
        + "=" * 80
    )

    # ==========================================
    # SECTION A : LOGISTICS PERFORMANCE
    # ==========================================
    # 1. Late Rate by Carrier
    carrier_df = (
        filtered_df.groupby("carrier")["late_delivery"].mean().reset_index()
    )
    carrier_df["late_delivery"] = carrier_df["late_delivery"] * 100
    fig_a1 = px.bar(
        carrier_df,
        x="late_delivery",
        y="carrier",
        orientation="h",
        title="Late Rate by Carrier (%)",
        labels={"late_delivery": "Late Rate (%)", "carrier": "Carrier"},
        color_discrete_sequence=["#2b5c8f"],
    )
    fig_a1.show()

    # 2. Temporal Evolution of Delays
    time_df = (
        filtered_df.groupby(filtered_df["order_date"].dt.to_period("M"))[
            "delivery_delay_days"
        ]
        .mean()
        .reset_index()
    )
    time_df["order_date"] = time_df["order_date"].astype(str)
    fig_a2 = px.line(
        time_df,
        x="order_date",
        y="delivery_delay_days",
        title="Temporal Evolution of Delays (Days)",
        labels={"order_date": "Month", "delivery_delay_days": "Avg Delay (Days)"},
    )
    fig_a2.show()

    # ==========================================
    # SECTION B : COSTS & CARRIERS
    # ==========================================
    # 1. Shipping Cost vs Distance
    fig_b1 = px.scatter(
        filtered_df.sample(min(len(filtered_df), 2000)),
        x="distance_km",
        y="shipping_cost_usd",
        color="carrier",
        title="Shipping Cost vs Distance",
        labels={
            "distance_km": "Distance (km)",
            "shipping_cost_usd": "Shipping Cost ($)",
        },
    )
    fig_b1.show()

    # 2. Shipping Methods Breakdown
    ship_method_df = (
        filtered_df["shipping_method"].value_counts().reset_index()
    )
    ship_method_df.columns = ["shipping_method", "count"]
    fig_b2 = px.pie(
        ship_method_df,
        names="shipping_method",
        values="count",
        title="Shipping Methods Breakdown",
        hole=0.4,
    )
    fig_b2.show()

    # ==========================================
    # SECTION C : RETURNS & CUSTOMER QUALITY
    # ==========================================
    # 1. Main Return Reasons
    returns_df = filtered_df[filtered_df["return_requested"] == True]
    if not returns_df.empty:
      return_reasons = returns_df["return_reason"].value_counts().reset_index()
      return_reasons.columns = ["return_reason", "count"]
      fig_c1 = px.bar(
          return_reasons,
          x="count",
          y="return_reason",
          orientation="h",
          title="Main Return Reasons",
          labels={"count": "Count", "return_reason": "Reason"},
          color_discrete_sequence=["#e74c3c"],
      )
      fig_c1.show()

    # 2. Customer Rating Histogram
    fig_c2 = px.histogram(
        filtered_df,
        x="customer_rating",
        nbins=5,
        title="Customer Rating Distribution (1 to 5 Stars)",
        labels={"customer_rating": "Rating", "count": "Frequency"},
        color_discrete_sequence=["#2ecc71"],
    )
    fig_c2.show()

    # ==========================================
    # SECTION D : EXTERNAL FACTORS
    # ==========================================
    # 1. Weather Impact on Delivery Attempts
    weather_df = (
        filtered_df.groupby("weather_condition")
        .agg(
            avg_attempts=("delivery_attempts", "mean"),
            on_time_rate=(
                "late_delivery",
                lambda x: (x == False).mean() * 100,
            ),
        )
        .reset_index()
    )

    fig_d1 = px.bar(
        weather_df,
        x="weather_condition",
        y="avg_attempts",
        title="Weather Impact on Average Delivery Attempts",
        labels={
            "weather_condition": "Weather",
            "avg_attempts": "Avg Delivery Attempts",
        },
        color_discrete_sequence=["#f39c12"],
    )
    fig_d1.show()

    # 2. Warehouse Processing Hours Boxplot
    fig_d2 = px.box(
        filtered_df,
        x="warehouse_id",
        y="warehouse_processing_hours",
        title="Warehouse Processing Hours by Warehouse",
        labels={
            "warehouse_id": "Warehouse ID",
            "warehouse_processing_hours": "Processing Hours",
        },
    )
    fig_d2.show()


# Link widgets to update function
category_dropdown.observe(update_dashboard, names="value")
shipping_dropdown.observe(update_dashboard, names="value")
warehouse_dropdown.observe(update_dashboard, names="value")

# Display dashboard control panel and output block
print("🎛️ GLOBAL FILTERS PANEL (Select/Deselect items to filter dashboard)")
display(
    widgets.HBox([category_dropdown, shipping_dropdown, warehouse_dropdown])
)
display(output)

# Initial trigger to render charts on load
update_dashboard(None)

🎛️ GLOBAL FILTERS PANEL (Select/Deselect items to filter dashboard)


Output()